In [7]:
import os
import json
import numpy as np

BASE_DIR = "data/sudoku-simple-10"
SPLIT = "train"   # "train" or "test"



In [8]:
def load_dataset(base_dir, split):
    split_dir = os.path.join(base_dir, split)

    data = {}
    for name in [
        "inputs",
        "labels",
        "group_indices",
        "puzzle_indices",
        "puzzle_identifiers",
    ]:
        data[name] = np.load(os.path.join(split_dir, f"all__{name}.npy"))

    with open(os.path.join(split_dir, "dataset.json")) as f:
        metadata = json.load(f)

    return data, metadata


data, metadata = load_dataset(BASE_DIR, SPLIT)


In [9]:
print("=== DATASET OVERVIEW ===")
print(f"Split: {SPLIT}")
print(f"Number of examples: {len(data['inputs'])}")
print(f"Inputs shape: {data['inputs'].shape}")
print(f"Labels shape: {data['labels'].shape}")

print("\n=== METADATA ===")
for k, v in metadata.items():
    print(f"{k}: {v}")


=== DATASET OVERVIEW ===
Split: train
Number of examples: 10
Inputs shape: (10, 81)
Labels shape: (10, 81)

=== METADATA ===
pad_id: 0
ignore_label_id: 0
blank_identifier_id: 0
vocab_size: 11
seq_len: 81
num_puzzle_identifiers: 1
total_groups: 10
mean_puzzle_examples: 1.0
total_puzzles: 10
sets: ['all']


In [10]:
def print_sudoku(grid):
    """
    Pretty-print a 9x9 Sudoku grid with 3x3 box separators.
    0 is displayed as '.'.
    """
    for r in range(9):
        row_str = []
        for c in range(9):
            val = grid[r, c]
            cell = "." if val == 0 else str(val)
            row_str.append(cell)

            # Vertical separators after columns 2 and 5
            if c in (2, 5):
                row_str.append("|")

        print(" ".join(row_str))

        # Horizontal separators after rows 2 and 5
        if r in (2, 5):
            print("-" * 21)


def show_example(data, index):
    puzzle = data["inputs"][index] - 1   # undo +1 offset
    solution = data["labels"][index] - 1

    puzzle = puzzle.reshape(9, 9)
    solution = solution.reshape(9, 9)

    print(f"\n=== Example {index} ===")
    print("\nPuzzle:")
    print_sudoku(puzzle)

    print("\nSolution:")
    print_sudoku(solution)

def show_examples(data, indices):
    for idx in indices:
        show_example(data, idx)
        print("\n" + "=" * 30 + "\n")

def show_group2(data, group_id):
    start = data["group_indices"][group_id]
    end = data["group_indices"][group_id + 1]

    print(f"Showing group {group_id} (examples {start} → {end - 1})")
    for idx in range(start, end):
        show_example(data, idx)
        print("\n" + "-" * 25 + "\n")

def show_group(data, group_id, n=None):
    """
    Show Sudoku examples from a group.

    Parameters
    ----------
    data : dict
        Loaded dataset dictionary
    group_id : int
        Group index to display
    n : int or None, optional
        Maximum number of examples to show from the group.
        If None, show all examples.
    """
    start = data["group_indices"][group_id]
    end = data["group_indices"][group_id + 1]

    total = end - start
    if n is not None:
        n = min(n, total)
        end = start + n

    print(
        f"Showing group {group_id} "
        f"(examples {start} → {end - 1}, total in group: {total})"
    )

    for idx in range(start, end):
        show_example(data, idx)
        print("\n" + "-" * 25 + "\n")




In [11]:
show_group(data, group_id=0, n=2)


Showing group 0 (examples 0 → 0, total in group: 1)

=== Example 0 ===

Puzzle:
2 . . | . 6 . | . 9 8
. . . | 3 9 . | 6 2 .
. . 8 | . . . | . 3 1
---------------------
. . . | 2 . 3 | 8 1 7
1 . . | 7 8 . | . . .
. . . | . . . | . . .
---------------------
. . . | 9 . . | 2 . .
9 5 6 | . 2 . | . . .
4 . . | 8 . 5 | . . .

Solution:
2 3 5 | 1 6 4 | 7 9 8
7 1 4 | 3 9 8 | 6 2 5
6 9 8 | 5 7 2 | 4 3 1
---------------------
5 6 9 | 2 4 3 | 8 1 7
1 4 3 | 7 8 9 | 5 6 2
8 7 2 | 6 5 1 | 3 4 9
---------------------
3 8 7 | 9 1 6 | 2 5 4
9 5 6 | 4 2 7 | 1 8 3
4 2 1 | 8 3 5 | 9 7 6

-------------------------



In [12]:
show_example(data, index=0)
show_examples(data, indices=[0, 1, 2])



=== Example 0 ===

Puzzle:
2 . . | . 6 . | . 9 8
. . . | 3 9 . | 6 2 .
. . 8 | . . . | . 3 1
---------------------
. . . | 2 . 3 | 8 1 7
1 . . | 7 8 . | . . .
. . . | . . . | . . .
---------------------
. . . | 9 . . | 2 . .
9 5 6 | . 2 . | . . .
4 . . | 8 . 5 | . . .

Solution:
2 3 5 | 1 6 4 | 7 9 8
7 1 4 | 3 9 8 | 6 2 5
6 9 8 | 5 7 2 | 4 3 1
---------------------
5 6 9 | 2 4 3 | 8 1 7
1 4 3 | 7 8 9 | 5 6 2
8 7 2 | 6 5 1 | 3 4 9
---------------------
3 8 7 | 9 1 6 | 2 5 4
9 5 6 | 4 2 7 | 1 8 3
4 2 1 | 8 3 5 | 9 7 6

=== Example 0 ===

Puzzle:
2 . . | . 6 . | . 9 8
. . . | 3 9 . | 6 2 .
. . 8 | . . . | . 3 1
---------------------
. . . | 2 . 3 | 8 1 7
1 . . | 7 8 . | . . .
. . . | . . . | . . .
---------------------
. . . | 9 . . | 2 . .
9 5 6 | . 2 . | . . .
4 . . | 8 . 5 | . . .

Solution:
2 3 5 | 1 6 4 | 7 9 8
7 1 4 | 3 9 8 | 6 2 5
6 9 8 | 5 7 2 | 4 3 1
---------------------
5 6 9 | 2 4 3 | 8 1 7
1 4 3 | 7 8 9 | 5 6 2
8 7 2 | 6 5 1 | 3 4 9
---------------------
3 8 7 | 9 1 6 | 2 5 